# Chapter 7 — Fine-tuning to follow instructions

Chapter 6 adapted pretrained GPT-2 representations to a fixed two-label classifier. Instruction fine-tuning keeps the language-model
output head and instead teaches the model to continue structured requests with useful natural-language responses.

The underlying objective remains next-token prediction. The difference is the supervised corpus: each example pairs an instruction and
optional task input with a desired response. This chapter prepares those examples, batches their token IDs, and fine-tunes the pretrained
model to reproduce the response pattern.

## 7.1 Downloading and validating instruction data

The book provides 1,100 supervised examples as JSON. Cache the file locally so notebook reruns avoid unnecessary network work, then
validate its external schema before the rest of the notebook relies on typed fields.

Each entry must contain three strings: `instruction`, `input`, and `output`. Validation catches truncated files or unexpected source
changes near the acquisition boundary.

In [1]:
import json
import urllib.request
from collections.abc import Sequence
from pathlib import Path
from typing import TypedDict, cast

import torch


class InstructionExample(TypedDict):
    """Schema for one supervised instruction-response example."""

    instruction: str
    input: str
    output: str


def download_and_load_file(
    file_path: str | Path,
    url: str,
) -> list[InstructionExample]:
    """Download and validate the instruction dataset when absent locally.

    Args:
        file_path: Local JSON path used as a persistent cache.
        url: Remote URL used only when the cache does not exist.

    Returns:
        Validated instruction examples with `instruction`, `input`, and
        `output` string fields.

    Raises:
        urllib.error.URLError: If the dataset download fails.
        UnicodeDecodeError: If downloaded bytes are not valid UTF-8.
        OSError: If reading or writing the local file fails.
        json.JSONDecodeError: If the local file is not valid JSON.
        ValueError: If the decoded JSON does not match the expected schema.
    """
    path = Path(file_path)
    if not path.exists():
        # The dataset is small enough to decode and write in one operation.
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        path.write_text(text_data, encoding="utf-8")

    raw_data = json.loads(path.read_text(encoding="utf-8"))
    required_fields = ("instruction", "input", "output")
    if not isinstance(raw_data, list) or not all(
        isinstance(entry, dict)
        and all(isinstance(entry.get(field), str) for field in required_fields)
        for entry in raw_data
    ):
        raise ValueError("Instruction data does not match the expected schema")
    # Runtime validation above makes this third-party JSON boundary safe to type.
    return cast(list[InstructionExample], raw_data)


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


## 7.2 Inspecting the instruction schema

An instruction describes the operation, optional input supplies data needed by that operation, and output contains the desired answer.
The spelling example uses all three fields.

In [2]:
# This example contains all three schema fields, including optional input.
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


### Instructions without additional input

Some tasks are fully specified by their instruction. Their `input` field is an empty string rather than a missing key, allowing every
entry to preserve one predictable schema.

In [3]:
# An empty input means the instruction is self-contained.
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


## 7.3 Formatting examples in Alpaca style

A consistent textual template tells the model which part describes the task, which part supplies optional data, and where its response
should begin. `format_input` deliberately omits the reference `output`; that answer is appended separately as the continuation the model
must learn to generate.

```text
Below is an instruction ...

### Instruction:
{instruction}

### Input:                 included only when nonempty
{input}

### Response:
{output}
```

In [4]:
def format_input(entry: InstructionExample) -> str:
    """Format an instruction and optional input without its answer.

    Args:
        entry: One validated instruction-response example.

    Returns:
        Alpaca-style model input ending before the response header.
    """
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    # Omit the complete section when no additional task input is required.
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

### Formatting an example with task input

The misspelled word appears under `### Input:`, while the corrected spelling follows `### Response:`. Keeping the desired response
separate from `model_input` will later make the prompt available for generation without revealing the reference answer.

In [5]:
# Format an example that requires an additional misspelled-word input.
model_input = format_input(data[50])
# Append the supervised continuation only after constructing the prompt.
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


### Formatting a self-contained instruction

When `input` is empty, omit its heading and move directly from the instruction to the response. This avoids teaching the model to expect
an unnecessary blank section.

In [6]:
# Self-contained instructions omit the `### Input:` section entirely.
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


## 7.4 Splitting instruction examples

Use 85% for parameter updates, 5% for validation, and 10% for final testing. Integer rounding assigns any remainder to validation so all
1,100 entries are retained exactly once.

This deterministic contiguous split follows the book and assumes the source ordering is sufficiently mixed. For a newly assembled or
ordered dataset, shuffle reproducibly before slicing or use a stratification strategy appropriate to its task categories.

In [7]:
# Reserve 85% for optimization, 10% for final testing, and the 5% remainder
# for validation-guided development.
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
val_portion = len(data) - train_portion - test_portion

test_start = train_portion
val_start = train_portion + test_portion
# Slices are non-overlapping, so each example belongs to exactly one split.
train_data = data[:train_portion]
test_data = data[test_start:val_start]
val_data = data[val_start:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## 7.5 Tokenizing complete supervised examples

`format_input` omitted the answer so prompts could be inspected independently. Training data now appends the `### Response:` header and
desired output, producing one continuous sequence for next-token prediction:

```text
formatted instruction + optional input + response header + desired output
```

`InstructionDataset` tokenizes every complete sequence once during initialization and stores variable-length Python lists. It does not
pad or convert them to tensors; a later collation function will create shifted batches and choose padding dynamically.

Eager tokenization avoids repeating tokenizer work every epoch, at the cost of retaining all encoded examples in host memory. The current
1,100-example corpus is small enough for that tradeoff.

In [8]:
import tiktoken
from torch.utils.data import Dataset


class InstructionDataset(Dataset[list[int]]):
    """Eagerly tokenize formatted instruction-response training examples."""

    def __init__(
        self,
        data: list[InstructionExample],
        tokenizer: tiktoken.Encoding,
    ) -> None:
        """Format and tokenize every example once during initialization.

        Args:
            data: Validated instruction-response examples in one dataset split.
            tokenizer: GPT-2 tokenizer used by the pretrained model.
        """
        self.data = data
        # Each inner list will contain one example's variable num_tokens.
        self.encoded_texts: list[list[int]] = []
        for entry in data:
            # The model input excludes the answer and may omit the input section.
            instruction_plus_input = format_input(entry)
            # The response header marks where the desired continuation begins.
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            # Tokenize the complete supervised sequence without padding yet.
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index: int) -> list[int]:
        """Return one variable-length instruction sequence as token IDs.

        Args:
            index: Zero-based position in this dataset split.

        Returns:
            Token IDs for one complete formatted example, conceptually shaped
            `(num_tokens,)` before tensor conversion and padding.

        Raises:
            IndexError: If `index` is outside the dataset.
        """
        return self.encoded_texts[index]

    def __len__(self) -> int:
        """Return the number of instruction-response examples.

        Returns:
            Number of formatted and tokenized examples in this split.
        """
        return len(self.data)

## 7.6 Reusing GPT-2's end-of-text token for padding

GPT-2 has no dedicated padding entry, so the book reuses `<|endoftext|>` with token ID `50256`. In this chapter it serves two related
roles: one meaningful target marks where a response should end, and repeated copies align shorter examples within a batch.

Those repeated targets must later be excluded from loss so long padding regions do not dominate learning.

In [9]:
# GPT-2 reuses its end-of-text vocabulary entry as the padding token.
tokenizer = tiktoken.get_encoding("gpt2")
endoftext_id = tokenizer.encode(
    "<|endoftext|>",
    allowed_special={"<|endoftext|>"},
)
print(endoftext_id)

[50256]


## 7.7 Draft 1: dynamically padding inputs

A data loader's collation function receives several variable-length `list[int]` examples. Determine the longest augmented sequence in
that batch, append one end-of-text ID to each example, and right-pad all examples to a common length.

Dynamic per-batch padding usually wastes fewer token positions than padding the complete dataset to one global maximum.

In [10]:
def custom_collate_draft_1(
    batch: Sequence[Sequence[int]],
    pad_token_id: int = 50256,
    device: str | torch.device = "cpu",
) -> torch.Tensor:
    """Right-pad variable-length token IDs into one input tensor.

    Args:
        batch: Variable-length token-ID sequences from `InstructionDataset`.
        pad_token_id: Token ID used for sequence termination and padding.
        device: Destination device for the stacked input tensor.

    Returns:
        Input token IDs shaped `(batch_size, num_tokens)`.

    Raises:
        ValueError: If `batch` is empty.
    """
    # Add one position so every sequence can predict an end-of-text target.
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst: list[torch.Tensor] = []

    for item in batch:
        new_item = list(item)
        new_item.append(pad_token_id)
        # Right-pad to the longest augmented sequence in this batch.
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        # Exclude the final position; Draft 2 will use it as a shifted target.
        inputs = torch.tensor(padded[:-1], dtype=torch.long)
        inputs_lst.append(inputs)

    # inputs_tensor: (batch_size, num_tokens=batch_max_length - 1)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

### Inspecting right padding

The five-token example determines this toy batch's width. Shorter examples retain their original IDs and receive `50256` on the right
until all three rows can be stacked into `(batch_size=3, num_tokens=5)`.

In [11]:
# Three artificial examples make right-padding behavior easy to inspect.
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1, inputs_2, inputs_3)
# Shape: (batch_size=3, num_tokens=5)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


## 7.8 Draft 2: creating shifted next-token targets

Use the same padded sequence to create inputs and targets offset by one position:

```text
padded   [t0, t1, t2, ..., end]
inputs   [t0, t1, t2, ...]
targets  [t1, t2, ..., end]
```

The appended end-of-text ID becomes a target, teaching the model to terminate after the desired response.

In [12]:
def custom_collate_draft_2(
    batch: Sequence[Sequence[int]],
    pad_token_id: int = 50256,
    device: str | torch.device = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Create padded next-token inputs and shifted targets.

    Args:
        batch: Variable-length token-ID sequences from `InstructionDataset`.
        pad_token_id: Token ID used for sequence termination and padding.
        device: Destination device for both stacked tensors.

    Returns:
        Input and target token IDs, each shaped
        `(batch_size, num_tokens)`.

    Raises:
        ValueError: If `batch` is empty.
    """
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst: list[torch.Tensor] = []
    targets_lst: list[torch.Tensor] = []

    for item in batch:
        new_item = list(item)
        new_item.append(pad_token_id)
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        # Shift by one position to create next-token prediction pairs.
        inputs = torch.tensor(padded[:-1], dtype=torch.long)
        targets = torch.tensor(padded[1:], dtype=torch.long)
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Both tensors: (batch_size, num_tokens=batch_max_length - 1)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor


inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


## 7.9 Ignoring repeated padding targets

Draft 2 asks short examples to predict the padding token repeatedly. The final collator keeps the first end-of-text target as a genuine
sequence-ending objective and replaces every later padding target with `ignore_index=-100`.

For one target row, `mask = targets == pad_token_id` has shape `(num_tokens,)`. `torch.nonzero(mask)` returns matching coordinates shaped
`(num_matches, 1)`, and `.squeeze()` removes the trailing size-one dimension. The resulting indices identify every end-of-text target.
`indices[1:]` deliberately skips the first match and selects only repeated padding targets.

PyTorch cross-entropy excludes `-100` positions by default. Padding IDs can remain in the input because tensor rows still need equal
length, but repeated padding contributes no direct target loss. `allowed_max_length` caps both inputs and targets together when a batch
would exceed model capacity.

In [13]:
def custom_collate_fn(
    batch: Sequence[list[int]],
    pad_token_id: int = 50256,
    ignore_index: int = -100,
    allowed_max_length: int | None = None,
    device: str | torch.device = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Create padded next-token inputs and loss-ready shifted targets.

    Args:
        batch: Variable-length token-ID lists from `InstructionDataset`.
        pad_token_id: Token ID used for sequence termination and padding.
        ignore_index: Target value excluded by cross-entropy loss.
        allowed_max_length: Optional maximum `num_tokens` retained per example.
        device: Destination device for both stacked tensors.

    Returns:
        Input and target tensors, each shaped `(batch_size, num_tokens)`.
        Repeated padding positions in targets contain `ignore_index`.

    Raises:
        ValueError: If `batch` is empty.
    """
    # Reserve one additional position for the end-of-text target.
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst: list[torch.Tensor]
    targets_lst: list[torch.Tensor]
    inputs_lst, targets_lst = [], []

    for item in batch:
        # Copy the dataset item so collation never mutates cached token IDs.
        new_item = item.copy()
        new_item += [pad_token_id]

        # All padded lists have length batch_max_length.
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        # Shift by one position to form next-token input-target pairs.
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # mask: (num_tokens,); nonzero coordinates identify end-of-text targets.
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            # Keep the first sequence-ending target and ignore later padding.
            targets[indices[1:]] = ignore_index

        # Truncate inputs and targets identically to preserve alignment.
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Both tensors: (batch_size, num_tokens)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

### Inspecting loss-ready targets

The longest example retains its end-of-text target. For shorter examples, the first `50256` remains trainable and later copies become
`-100`. Inputs contain no `-100` values because ignore indices belong only in the target tensor consumed by cross-entropy.

In [14]:
inputs, targets = custom_collate_fn(batch)
# Both tensors: (batch_size=3, num_tokens=5)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


## 7.10 Verifying `ignore_index` in cross-entropy

A small two-token vocabulary makes loss averaging visible. The first example has two target positions: one incorrect high-confidence
prediction and one correct high-confidence prediction. Cross-entropy averages their negative log probabilities into one scalar.

In [15]:
# logits_1: (num_tokens=2, toy_vocab_size=2)
logits_1 = torch.tensor([[-1.0, 1.0], [-0.5, 1.5]])
# targets_1: (num_tokens=2,), one desired vocabulary ID per position
targets_1 = torch.tensor([0, 1])  # Correct token indices to generate
# loss_1: scalar tensor shaped (), averaged across both target positions
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


### Including an additional target position

Add a third easy prediction and include it in the targets. Because the loss is a mean over non-ignored positions, this extra low-loss
position reduces the reported average.

In [16]:
# Add a third, confidently correct token prediction.
# logits_2: (num_tokens=3, toy_vocab_size=2)
logits_2 = torch.tensor([[-1.0, 1.0], [-0.5, 1.5], [-0.5, 1.5]])
# targets_2: (num_tokens=3,)
targets_2 = torch.tensor([0, 1, 1])
# The easy third position lowers the mean loss when it is included.
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


### Excluding a target with `-100`

Replace the third target with `-100`. PyTorch omits that position from both the loss sum and denominator, making the result equal to the
original two-position loss. This is exactly how repeated padding targets are prevented from improving loss artificially.

In [17]:
# Replace the third target with cross-entropy's default ignore index.
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
# Ignoring position three reproduces the original two-position mean exactly.
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


## 7.11 Selecting the training device

Use CUDA when available because instruction fine-tuning requires repeated Transformer forward and backward passes. CPU remains a
functional fallback. Model parameters and collated input and target tensors must ultimately share this device.

In [18]:
# Prefer an NVIDIA GPU and fall back to CPU when CUDA is unavailable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Apple Silicon users can optionally select the MPS backend instead.
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
print("Device:", device)

Device: cuda


### Binding collation settings

`functools.partial` creates a callable that still accepts `batch` but automatically supplies the selected device and the 1,024-token GPT-2
context limit. `DataLoader` can invoke that callable without knowing about run-wide configuration.

In [19]:
from functools import partial

# Bind run-wide settings while leaving only `batch` for DataLoader to supply.
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024,
)

## 7.12 Building instruction data loaders

Each loader combines an `InstructionDataset` with the customized collator and yields:

```text
input_batch   (batch_size, num_tokens)
target_batch  (batch_size, num_tokens)
```

Shuffle training examples and discard its incomplete remainder for fixed-size optimizer steps. Keep validation and test order stable and
retain their final incomplete batches so every held-out example contributes to evaluation.

`num_workers=0` performs collation in the notebook process. This is especially appropriate here because the collator moves tensors to
CUDA; creating CUDA tensors in worker subprocesses is fragile and generally discouraged.

In [20]:
from torch.utils.data import DataLoader

# Keeping collation in the notebook process is reliable when it creates CUDA tensors.
num_workers = 0
batch_size = 8

# Reproduce the shuffled order of training examples.
torch.manual_seed(123)

# PyTorch's DataLoader type parameter describes dataset items, not the value
# returned by a transforming collate_fn. Leave the loader variables inferred;
# custom_collate_fn already documents the actual tensor-tuple batch contract.
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    # Keep every optimizer step at the configured batch_size.
    drop_last=True,
    num_workers=num_workers,
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    # Retain all held-out examples for evaluation.
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

## Chapter 7 summary

The chapter now prepares complete instruction-tuning loaders:

- dynamic collation pads and shifts variable-length examples;
- `ignore_index=-100` removes repeated padding targets from cross-entropy;
- a partial collator binds model capacity and device settings;
- training batches shuffle and keep a fixed size; and
- validation and test loaders retain every held-out example.

The loaders yield input and target tensors shaped `(batch_size, num_tokens)` directly on the selected device. Their targets preserve one
end-of-text objective per example while excluding artificial padding from loss.

In [21]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.

In [22]:
from gpt_download import download_and_load_gpt2

from build_llms_from_scratch_companion.model import GPTConfig, GPTModel
from build_llms_from_scratch_companion.training import load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Context length
    "dropout_rate": 0.0,  # Dropout rate
    "qkv_bias": True,  # Query-key-value bias
    "num_heads": 16,
    "num_layers": 24,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")
cfg = GPTConfig(**BASE_CONFIG)
model = GPTModel(cfg)
load_weights_into_gpt(model, params)
model.eval();

File already exists and is up-to-date: gpt2\355M\checkpoint
File already exists and is up-to-date: gpt2\355M\encoder.json
File already exists and is up-to-date: gpt2\355M\hparams.json
File already exists and is up-to-date: gpt2\355M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\355M\model.ckpt.index
File already exists and is up-to-date: gpt2\355M\model.ckpt.meta
File already exists and is up-to-date: gpt2\355M\vocab.bpe


In [23]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [25]:
from build_llms_from_scratch_companion.training import (
    generate,
    text_to_token_ids,
    token_ids_to_text,
)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=cfg.context_length,
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
generated_text

"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nConvert the active sentence to passive: 'The chef cooks the meal every day.'\n\n### Response:\n\nThe chef cooks the meal every day.\n\n### Instruction:\n\nConvert the active sentence to passive: 'The chef cooks the"